# Import Libraries

In [1]:
import torch
from torchvision import transforms as T
from utils.set_seed import set_random_seed
from torch.utils.tensorboard import SummaryWriter

device = 'cpu'
if torch.backends.mps.is_available(): device = torch.device('mps')
elif torch.cuda.is_available(): device = torch.device('cuda')

In [2]:
print(device)

cuda


In [3]:
from weathernet import WeatherNet
from weathernetplusplus import WeatherNetPlusPlus
from mtl_weathernet import MtlWeatherNet
from weathernet_transformer import WeatherNetTransformer

from data.data import get_dataloaders
import utils.trainer as trainer

# Set Hyperparameters, Load Dataset

In [4]:
# Hyperparameters
# TODO: experiment with different hyperparameters
batch_size = 32
learning_rate = 0.001
epochs = 10

set_random_seed(42) # seed for reproducibility

In [5]:
# Load BDD100KPlus dataset
trainloader, valloader, testloader = get_dataloaders(dataset_name="Bdd100kPlus", batch_size=batch_size)

In [6]:
# Print dataset statistics
print(f"Number of training samples: {len(trainloader.dataset)}")
print(f"Number of validation samples: {len(valloader.dataset)}")
print(f"Number of test samples: {len(testloader.dataset)}")

# print batches
print(f"Number of batches in training set: {len(trainloader)}")
print(f"Number of batches in validation set: {len(valloader)}")
print(f"Number of batches in test set: {len(testloader)}")

Number of training samples: 70000
Number of validation samples: 10000
Number of test samples: 20000
Number of batches in training set: 2188
Number of batches in validation set: 313
Number of batches in test set: 625


# Initialize WeatherNet Model

In [7]:
# wn_model: WeatherNet = WeatherNet()
# wn_model: WeatherNetPlusPlus = WeatherNetPlusPlus()
# wn_model: MtlWeatherNet = MtlWeatherNet()
wn_model: WeatherNetTransformer = WeatherNetTransformer()

# Print the model architecture
print(wn_model)

# Define optimizer for the model
# Could explore SGD, Adam, AdamW, etc
optimizer = torch.optim.Adam(wn_model.parameters())

# Define loss function
criterion = torch.nn.CrossEntropyLoss()

WeatherNetTransformer(
  (backbone): VisionTransformer(
    (conv_proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (encoder): Encoder(
      (dropout): Dropout(p=0.0, inplace=False)
      (layers): Sequential(
        (encoder_layer_0): EncoderBlock(
          (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (self_attention): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (dropout): Dropout(p=0.0, inplace=False)
          (ln_2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (mlp): MLPBlock(
            (0): Linear(in_features=768, out_features=3072, bias=True)
            (1): GELU(approximate='none')
            (2): Dropout(p=0.0, inplace=False)
            (3): Linear(in_features=3072, out_features=768, bias=True)
            (4): Dropout(p=0.0, inplace=False)
          )
        )
        (encoder_layer_1): EncoderBlock(
        

In [8]:
saved_state = None # path to saved model state
if saved_state is not None:
    print(f"Loading saved state from {saved_state}")
    wn_model.load_state_dict(torch.load(saved_state, weights_only=True))

# Model Training

In [9]:
# Train the WeatherNet model and record losses
# Use the train function to train the wn_model with optimizer on the trainloader for a specified number of epochs (e.g., 5)
# Record the training and test losses in wn_train_losses and wn_test_losses respectively, pass hyperparameters

# logs to runs/WeatherNet/ or runs/WeatherNetPlusPlus/ or runs/MtlWeatherNet/
writer = SummaryWriter(log_dir=f"runs/{wn_model.name}")

# epochs=1
wn_train_loss_log, wn_test_loss_log = trainer.train(wn_model, optimizer, criterion, trainloader, valloader, epochs, device, "./checkpoints", writer=writer)

writer.flush()

Begin training
Epoch 1/10
* Batch 10/2188 - fog loss: 0.00 | glare loss: 0.46 | road loss: 0.38 | traffic loss: 0.43 | weather loss: 1.29 | scene loss: 0.71 | tod loss: 0.62
* Batch 20/2188 - fog loss: 0.01 | glare loss: 0.72 | road loss: 0.47 | traffic loss: 0.19 | weather loss: 1.22 | scene loss: 0.94 | tod loss: 0.37
* Batch 30/2188 - fog loss: 0.01 | glare loss: 0.53 | road loss: 0.54 | traffic loss: 0.12 | weather loss: 1.07 | scene loss: 0.86 | tod loss: 0.34
* Batch 40/2188 - fog loss: 0.03 | glare loss: 0.40 | road loss: 0.63 | traffic loss: 0.42 | weather loss: 1.29 | scene loss: 0.76 | tod loss: 0.39
* Batch 50/2188 - fog loss: 0.00 | glare loss: 0.34 | road loss: 0.38 | traffic loss: 0.40 | weather loss: 1.04 | scene loss: 0.99 | tod loss: 0.19
* Batch 60/2188 - fog loss: 0.00 | glare loss: 0.42 | road loss: 0.82 | traffic loss: 0.41 | weather loss: 1.65 | scene loss: 1.01 | tod loss: 0.38
* Batch 70/2188 - fog loss: 0.00 | glare loss: 0.42 | road loss: 0.53 | traffic loss: 

TypeError: cannot unpack non-iterable NoneType object

# Visualize Results

First, ensure you're in the correct environment:

`conda env create -f environment.yml`

This will create a conda environment called *tensorboard*, which you can activate via `conda activate tensorboard`

Then, to visualize the results: 

`tensorboard --logdir=runs`

This will automatically and recursively scan through all run logs in the runs/ directory.